In [2]:
import os
import numpy as np
import pandas as pd

# ======================================================
# PATHS (EXPLICIT, REVIEWER-SAFE)
# ======================================================

train_path = r"E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_train.csv"
test_path  = r"E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_test.csv"

# ======================================================
# CONSTANTS AND DEFINITIONS
# ======================================================

EARLY_MONTHS = [1, 2, 3, 4]
PEAK_MONTHS  = [5, 6, 7, 8]
LATE_MONTHS  = [9, 10, 11]

HEAT_THRESHOLD = 30.0  # °C

VEG_VARS  = ["NDVI", "EVI", "GPP"]
CLIM_VARS = ["PPT", "TMEAN", "TMIN", "TMAX", "TDMEAN", "VPDMIN", "VPDMAX"]

ANOMALY_BASE_COLS = [
    "NDVI_season_mean",
    "PPT_season_sum",
    "TMEAN_season_mean"
]

# ======================================================
# HELPER FUNCTIONS
# ======================================================

def monthly_columns(df, var):
    return [f"{var}_{i}" for i in range(1, 12) if f"{var}_{i}" in df.columns]


def fill_monthly_nans(df, cols, fill_values):
    for c in cols:
        df[c] = df[c].fillna(fill_values[c])
    return df


def seasonal_aggregates(df, cols, prefix):
    df[f"{prefix}_mean"] = df[cols].mean(axis=1)
    df[f"{prefix}_sum"]  = df[cols].sum(axis=1)
    df[f"{prefix}_max"]  = df[cols].max(axis=1)
    df[f"{prefix}_std"]  = df[cols].std(axis=1)
    return df


def phase_mean(df, var, months, phase):
    cols = [f"{var}_{m}" for m in months if f"{var}_{m}" in df.columns]
    df[f"{var}_{phase}_mean"] = df[cols].mean(axis=1)
    return df


# ======================================================
# LOAD DATA
# ======================================================

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# ======================================================
# 1. HANDLE NaNs IN MONTHLY VARIABLES (TRAIN-BASED)
# ======================================================

monthly_fill_values = {}

for var in VEG_VARS + CLIM_VARS:
    cols = monthly_columns(train_df, var)
    for c in cols:
        monthly_fill_values[c] = train_df[c].mean()

train_df = fill_monthly_nans(train_df, monthly_fill_values.keys(), monthly_fill_values)
test_df  = fill_monthly_nans(test_df,  monthly_fill_values.keys(), monthly_fill_values)

# ======================================================
# 2. SEASONAL AGGREGATES
# ======================================================

for var in VEG_VARS + CLIM_VARS:
    cols = monthly_columns(train_df, var)
    if cols:
        train_df = seasonal_aggregates(train_df, cols, f"{var}_season")
        test_df  = seasonal_aggregates(test_df,  cols, f"{var}_season")

# ======================================================
# 3. PHENOLOGICAL FEATURES
# ======================================================

for var in ["NDVI", "GPP", "PPT", "TMEAN"]:
    for df in [train_df, test_df]:
        df = phase_mean(df, var, EARLY_MONTHS, "early")
        df = phase_mean(df, var, PEAK_MONTHS,  "peak")
        df = phase_mean(df, var, LATE_MONTHS,  "late")

# ======================================================
# 4. STRESS INDICATORS
# ======================================================

tmax_cols = monthly_columns(train_df, "TMAX")
tmin_cols = monthly_columns(train_df, "TMIN")

for df in [train_df, test_df]:
    df["T_range_season_mean"] = df[tmax_cols].mean(axis=1) - df[tmin_cols].mean(axis=1)
    df["Heat_stress_months"]  = (df[tmax_cols] > HEAT_THRESHOLD).sum(axis=1)

# ======================================================
# 5. EFFICIENCY METRICS
# ======================================================

for df in [train_df, test_df]:
    df["NDVI_PPT_efficiency"] = df["NDVI_season_mean"] / (df["PPT_season_sum"] + 1e-6)
    df["GPP_PPT_efficiency"]  = df["GPP_season_sum"]  / (df["PPT_season_sum"] + 1e-6)

# ======================================================
# 6. TRAIN-BASED ANOMALY FEATURES (NO LEAKAGE)
# ======================================================

anomaly_stats = {}

for col in ANOMALY_BASE_COLS:
    stats = train_df.groupby("GEOID")[col].agg(["mean", "std"])
    anomaly_stats[col] = stats

def apply_anomalies(df, stats_dict):
    for col, stats in stats_dict.items():
        mean = df["GEOID"].map(stats["mean"])
        std  = df["GEOID"].map(stats["std"])
        global_mean = stats["mean"].mean()
        global_std  = stats["std"].mean()

        mean = mean.fillna(global_mean)
        std  = std.fillna(global_std)

        df[f"{col}_z"] = (df[col] - mean) / (std + 1e-6)
    return df

train_df = apply_anomalies(train_df, anomaly_stats)
test_df  = apply_anomalies(test_df,  anomaly_stats)

# ======================================================
# 7. SAVE OUTPUT FILES
# ======================================================

train_out = os.path.join(os.path.dirname(train_path), "yield_train_FE.csv")
test_out  = os.path.join(os.path.dirname(test_path),  "yield_test_FE.csv")

train_df.to_csv(train_out, index=False)
test_df.to_csv(test_out, index=False)

print("Feature engineering completed successfully.")
print("Saved:")
print(train_out)
print(test_out)


C:\Users\hp\AppData\Local\Temp\ipykernel_8592\1009656857.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{prefix}_mean"] = df[cols].mean(axis=1)
C:\Users\hp\AppData\Local\Temp\ipykernel_8592\1009656857.py:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{prefix}_sum"]  = df[cols].sum(axis=1)
C:\Users\hp\AppData\Local\Temp\ipykernel_8592\1009656857.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

Feature engineering completed successfully.
Saved:
E:/Abroad period research/New idea for 2026/Corn Yield Estimation\yield_train_FE.csv
E:/Abroad period research/New idea for 2026/Corn Yield Estimation\yield_test_FE.csv


Optimized feature set after ablation results

In [5]:
import os
import numpy as np
import pandas as pd

# ======================================================
# PATHS (EXPLICIT, REVIEWER-SAFE)
# ======================================================

train_path = r"E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_train.csv"
test_path  = r"E:/Abroad period research/New idea for 2026/Corn Yield Estimation/yield_test.csv"

# ======================================================
# CONSTANTS AND DEFINITIONS
# ======================================================

EARLY_MONTHS = [1, 2, 3, 4]
PEAK_MONTHS  = [5, 6, 7, 8]
LATE_MONTHS  = [9, 10, 11]

HEAT_THRESHOLD = 30.0  # °C

VEG_VARS  = ["NDVI", "EVI", "GPP"]
CLIM_VARS = ["PPT", "TMEAN", "TMIN", "TMAX", "TDMEAN", "VPDMIN", "VPDMAX"]

ANOMALY_BASE_COLS = [
    "NDVI_season_mean",
    "PPT_season_sum",
    "TMEAN_season_mean"
]

# ======================================================
# HELPER FUNCTIONS
# ======================================================

def monthly_columns(df, var):
    return [f"{var}_{i}" for i in range(1, 12) if f"{var}_{i}" in df.columns]


def fill_monthly_nans(df, cols, fill_values):
    for c in cols:
        df[c] = df[c].fillna(fill_values[c])
    return df


def seasonal_aggregates(df, cols, prefix):
    df[f"{prefix}_mean"] = df[cols].mean(axis=1)
    df[f"{prefix}_sum"]  = df[cols].sum(axis=1)
    df[f"{prefix}_max"]  = df[cols].max(axis=1)
    df[f"{prefix}_std"]  = df[cols].std(axis=1)
    return df


def phase_mean(df, var, months, phase):
    cols = [f"{var}_{m}" for m in months if f"{var}_{m}" in df.columns]
    df[f"{var}_{phase}_mean"] = df[cols].mean(axis=1)
    return df


# ======================================================
# LOAD DATA
# ======================================================

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# ======================================================
# 1. HANDLE NaNs IN MONTHLY VARIABLES (TRAIN-BASED)
# ======================================================

monthly_fill_values = {}

for var in VEG_VARS + CLIM_VARS:
    cols = monthly_columns(train_df, var)
    for c in cols:
        monthly_fill_values[c] = train_df[c].mean()

train_df = fill_monthly_nans(train_df, monthly_fill_values.keys(), monthly_fill_values)
test_df  = fill_monthly_nans(test_df,  monthly_fill_values.keys(), monthly_fill_values)

# ======================================================
# 2. SEASONAL AGGREGATES
# ======================================================

for var in VEG_VARS + CLIM_VARS:
    cols = monthly_columns(train_df, var)
    if cols:
        train_df = seasonal_aggregates(train_df, cols, f"{var}_season")
        test_df  = seasonal_aggregates(test_df,  cols, f"{var}_season")

# # ======================================================
# # 3. PHENOLOGICAL FEATURES
# # ======================================================

# for var in ["NDVI", "GPP", "PPT", "TMEAN"]:
#     for df in [train_df, test_df]:
#         df = phase_mean(df, var, EARLY_MONTHS, "early")
#         df = phase_mean(df, var, PEAK_MONTHS,  "peak")
#         df = phase_mean(df, var, LATE_MONTHS,  "late")

# # ======================================================
# # 4. STRESS INDICATORS
# # ======================================================

# tmax_cols = monthly_columns(train_df, "TMAX")
# tmin_cols = monthly_columns(train_df, "TMIN")

# for df in [train_df, test_df]:
#     df["T_range_season_mean"] = df[tmax_cols].mean(axis=1) - df[tmin_cols].mean(axis=1)
#     df["Heat_stress_months"]  = (df[tmax_cols] > HEAT_THRESHOLD).sum(axis=1)

# # ======================================================
# # 5. EFFICIENCY METRICS
# # ======================================================

# for df in [train_df, test_df]:
#     df["NDVI_PPT_efficiency"] = df["NDVI_season_mean"] / (df["PPT_season_sum"] + 1e-6)
#     df["GPP_PPT_efficiency"]  = df["GPP_season_sum"]  / (df["PPT_season_sum"] + 1e-6)

# ======================================================
# 6. TRAIN-BASED ANOMALY FEATURES (NO LEAKAGE)
# ======================================================

# anomaly_stats = {}

# for col in ANOMALY_BASE_COLS:
#     stats = train_df.groupby("GEOID")[col].agg(["mean", "std"])
#     anomaly_stats[col] = stats

# def apply_anomalies(df, stats_dict):
#     for col, stats in stats_dict.items():
#         mean = df["GEOID"].map(stats["mean"])
#         std  = df["GEOID"].map(stats["std"])
#         global_mean = stats["mean"].mean()
#         global_std  = stats["std"].mean()

#         mean = mean.fillna(global_mean)
#         std  = std.fillna(global_std)

#         df[f"{col}_z"] = (df[col] - mean) / (std + 1e-6)
#     return df

# train_df = apply_anomalies(train_df, anomaly_stats)
# test_df  = apply_anomalies(test_df,  anomaly_stats)

# ======================================================
# 7. SAVE OUTPUT FILES
# ======================================================

train_out = os.path.join(os.path.dirname(train_path), "yield_train_FE_AAO.csv")
test_out  = os.path.join(os.path.dirname(test_path),  "yield_test_FE_AAO.csv")

train_df.to_csv(train_out, index=False)
test_df.to_csv(test_out, index=False)

print("Feature engineering completed successfully.")
print("Saved:")
print(train_out)
print(test_out)


C:\Users\hp\AppData\Local\Temp\ipykernel_8592\1433775594.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{prefix}_mean"] = df[cols].mean(axis=1)
C:\Users\hp\AppData\Local\Temp\ipykernel_8592\1433775594.py:47: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{prefix}_sum"]  = df[cols].sum(axis=1)
C:\Users\hp\AppData\Local\Temp\ipykernel_8592\1433775594.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

Feature engineering completed successfully.
Saved:
E:/Abroad period research/New idea for 2026/Corn Yield Estimation\yield_train_FE_AAO.csv
E:/Abroad period research/New idea for 2026/Corn Yield Estimation\yield_test_FE_AAO.csv
